# FlowEdit Full Evaluation Colab

Evaluate the FlowEdit released dataset with the same sample subset and prompts for:

- Original FlowEdit baseline
- bridge_interpolate
- bridge_directional

The default run is intentionally small. Set `sample_limit_choice = "full"` only when you are ready for the full 281-pair run.


## Setup


In [ ]:
# Configuration
import os
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "flowedit-full-eval"
WORKDIR = "/content/FlowEdit"

model_name = "sd3"  # "sd3" or "flux"
methods_to_run = ["flowedit_baseline", "bridge_interpolate", "bridge_directional"]
budget_mode = "paper"  # "paper" follows FlowEdit paper T/n_max; "same_nfe" halves bridge edit steps.

DATASET_YAML = "Data/flowedit.yaml"
OUTPUT_ROOT = "outputs/flowedit_eval"
PIPELINE_LOAD_MODE = "auto"

sample_limit_choice = 10  # 3, 10, 20, 50, 100, or "full"
sample_limit = None if str(sample_limit_choice).lower() == "full" else int(sample_limit_choice)
sample_offset = 0

force_rerun = False
force_metrics = False
RUN_MODEL_PREFLIGHT = False
NUM_QUALITATIVE_EXAMPLES = min(5, sample_limit or 5)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

print("Branch:", BRANCH)
print("Model:", model_name)
print("Methods:", methods_to_run)
print("Sample limit:", sample_limit if sample_limit is not None else "full")
print("Force rerun:", force_rerun)


In [ ]:
# Clone repository, install dependencies, and authenticate if needed.
import os
import subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError("Could not checkout the evaluation branch. Push the branch first, then rerun this cell.")
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=False)

subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "plotly==5.24.1",
    "matplotlib",
    "diffusers>=0.31.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "safetensors",
    "sentencepiece",
    "einops",
    "pyyaml",
    "huggingface_hub",
    "lpips",
    "dreamsim",
], check=True)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF token provided. Public downloads only.")


## Load Model: SD3 or FLUX


In [ ]:
# Build the three-method experiment YAML for the selected model.
import subprocess
from pathlib import Path
import pandas as pd
import yaml

MODEL_ROOT = Path(OUTPUT_ROOT) / model_name
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
EXP_YAML = MODEL_ROOT / f"{model_name}_full_eval_config.yaml"
RUN_SUMMARY_CSV = MODEL_ROOT / "run_summary.csv"

subprocess.run([
    "python", "flowedit_eval.py", "write-config",
    "--model_name", model_name,
    "--dataset_yaml", DATASET_YAML,
    "--methods", ",".join(methods_to_run),
    "--budget_mode", budget_mode,
    "--output_yaml", str(EXP_YAML),
], check=True)

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)

display(pd.DataFrame(exp)[[
    "exp_name", "method_name", "solver_type", "model_type", "T_steps", "n_max",
    "src_guidance_scale", "tar_guidance_scale", "seed",
]])

if RUN_MODEL_PREFLIGHT:
    subprocess.run([
        "python", "run_script.py",
        "--exp_yaml", str(EXP_YAML),
        "--pipeline_load_mode", PIPELINE_LOAD_MODE,
        "--preflight_only",
    ], check=True)


## Load FlowEdit Dataset


In [ ]:
# Inspect the released FlowEdit dataset and selected sample subset.
from flowedit_eval import load_eval_samples
import pandas as pd

all_samples = load_eval_samples(DATASET_YAML)
selected_samples = load_eval_samples(
    DATASET_YAML,
    sample_limit=sample_limit,
    sample_offset=sample_offset,
)

print("Full dataset pairs:", len(all_samples))
print("Full dataset images:", len({s.image_id for s in all_samples}))
print("Selected pairs:", len(selected_samples))

display(pd.DataFrame([s.__dict__ for s in selected_samples]).head(20))


## Choose Sample Limit and Methods


In [ ]:
# This cell records the exact run choices before editing starts.
run_config = {
    "model_name": model_name,
    "methods_to_run": methods_to_run,
    "budget_mode": budget_mode,
    "dataset_yaml": DATASET_YAML,
    "sample_limit": sample_limit if sample_limit is not None else "full",
    "sample_offset": sample_offset,
    "force_rerun": force_rerun,
    "force_metrics": force_metrics,
    "output_root": OUTPUT_ROOT,
}
display(pd.DataFrame([run_config]))


## Run Editing


In [ ]:
# Generate edited images. Existing images are skipped unless force_rerun=True.
import subprocess
import pandas as pd

cmd = [
    "python", "run_script.py",
    "--device_number", "0",
    "--exp_yaml", str(EXP_YAML),
    "--dataset_yaml", DATASET_YAML,
    "--eval_output_root", OUTPUT_ROOT,
    "--run_summary_csv", str(RUN_SUMMARY_CSV),
    "--pipeline_load_mode", PIPELINE_LOAD_MODE,
    "--sample_offset", str(sample_offset),
]
if sample_limit is not None:
    cmd += ["--sample_limit", str(sample_limit)]
if force_rerun:
    cmd += ["--force_rerun"]

print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

run_summary = pd.read_csv(RUN_SUMMARY_CSV)
display(run_summary[[
    "sample_id", "method", "solver_type", "actual_nfe", "elapsed_seconds",
    "cached_generation", "output_image",
]].head(30))
print("Rows:", len(run_summary))


## Compute Metrics


In [ ]:
# Compute FlowEdit paper metrics: CLIP-T, CLIP-I, LPIPS, DINO, DreamSim.
import subprocess

metrics_cmd = [
    "python", "flowedit_eval.py", "metrics",
    "--run_summary_csv", str(RUN_SUMMARY_CSV),
    "--output_root", OUTPUT_ROOT,
    "--model_name", model_name,
]
if force_metrics:
    metrics_cmd += ["--force_metrics"]

print("$", " ".join(metrics_cmd))
subprocess.run(metrics_cmd, check=True)


## Generate Plots and Tables


In [ ]:
# Display summary, delta, win-rate tables, and the CLIP-T vs LPIPS plot.
from IPython.display import Image as IPImage, display
import pandas as pd
from pathlib import Path

SUMMARY_CSV = MODEL_ROOT / "summary_metrics.csv"
DELTA_CSV = MODEL_ROOT / "delta_vs_flowedit_baseline.csv"
WIN_RATE_CSV = MODEL_ROOT / "win_rate_vs_flowedit_baseline.csv"
PLOT_PNG = MODEL_ROOT / "plots" / "clip_t_vs_lpips.png"

summary = pd.read_csv(SUMMARY_CSV)
delta = pd.read_csv(DELTA_CSV)
win_rate = pd.read_csv(WIN_RATE_CSV)

print("Summary metrics")
display(summary)
print("Delta vs FlowEdit baseline. Positive means improvement under each metric direction.")
display(delta)
print("Win rate vs FlowEdit baseline")
display(win_rate)

if PLOT_PNG.exists():
    display(IPImage(filename=str(PLOT_PNG)))
else:
    print("Plot not found:", PLOT_PNG)


## Display Qualitative Comparison Examples


In [ ]:
# Show source and edited images side by side for a few selected pairs.
import base64
import html
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

metrics = pd.read_csv(MODEL_ROOT / "all_metrics_per_pair.csv")
examples = list(dict.fromkeys(metrics["sample_id"].tolist()))[:NUM_QUALITATIVE_EXAMPLES]
methods = [m for m in methods_to_run if m in set(metrics["method"])]

def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "") or "png"
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"

def img_tag(path, width=180):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border:1px solid #ddd;'>"

rows_html = []
for sample_id in examples:
    sample_rows = metrics[metrics["sample_id"] == sample_id]
    first = sample_rows.iloc[0]
    cells = [
        "<td>" +
        f"<b>{html.escape(sample_id)}</b><br>" +
        img_tag(first["source_image_path"]) +
        f"<br><small>{html.escape(str(first['target_prompt'])[:180])}</small>" +
        "</td>"
    ]
    for method in methods:
        row_df = sample_rows[sample_rows["method"] == method]
        if row_df.empty:
            cells.append(f"<td><b>{html.escape(method)}</b><br><em>missing</em></td>")
            continue
        row = row_df.iloc[0]
        metric_line = (
            f"CLIP-T {float(row['CLIP-T']):.3f} | "
            f"LPIPS {float(row['LPIPS']):.3f} | "
            f"DINO {float(row['DINO']):.3f}"
        )
        cells.append(
            "<td>" +
            f"<b>{html.escape(method)}</b><br>" +
            img_tag(row["output_image"]) +
            f"<br><small>{metric_line}</small>" +
            "</td>"
        )
    rows_html.append("<tr>" + "".join(cells) + "</tr>")

headers = "<th>Source / target prompt</th>" + "".join(f"<th>{html.escape(m)}</th>" for m in methods)
table_html = (
    "<table style='border-collapse:collapse;width:100%;'>"
    f"<thead><tr>{headers}</tr></thead>"
    "<tbody>" + "".join(rows_html) + "</tbody></table>"
)
display(HTML(table_html))
